# ДЗ 8. Мониторинг ML-систем

## 1. Ключевые метрики ML системы

Главная целевая метрика - выручка сервиса.

Бизнес метрики 1 уровня:
- Conversion Rate
- User Engagement
- Retention

ML метрики 2 уровня:
- Precision@K
- Recall@K
- Diversity
- Churn Rate

DevOps метрики 3 уровня:
- Latency p95
- Error Rate
- Availability 

Инфраструктурные метрики 4 уровня:
- RPS
- Inference Time
- Deploy Success Rate

## 2. Настройка мониторинга

Были настроены prometheus и graphana для мониторинга, а также написан простой ml сервис:

1. Поднимем всю необходимую инфраструктуру при помощи [docker-compose.yaml](docker-compose.yaml):

In [19]:
!docker compose down
!docker compose up --build -d

[+] down 2/4
 ✔ Container mipt-ml-monitoring-dqops-1      Removed                        0.0s
 ⠋ Container mipt-ml-monitoring-ml-service-1 Stopping                       0.1s
 ⠋ Container mipt-ml-monitoring-grafana-1    Stopping                       0.1s
 ✔ Container mipt-ml-monitoring-postgres-1   Removed                        0.0s
[+] down 2/4
 ✔ Container mipt-ml-monitoring-dqops-1      Removed                        0.0s
 ⠙ Container mipt-ml-monitoring-ml-service-1 Stopping                       0.2s
 ⠙ Container mipt-ml-monitoring-grafana-1    Stopping                       0.2s
 ✔ Container mipt-ml-monitoring-postgres-1   Removed                        0.0s
[+] down 2/4
 ✔ Container mipt-ml-monitoring-dqops-1      Removed                        0.0s
 ⠹ Container mipt-ml-monitoring-ml-service-1 Stopping                       0.3s
 ⠹ Container mipt-ml-monitoring-grafana-1    Removing                       0.3s
 ✔ Container mipt-ml-monitoring-postgres-1   Removed                  

2. На ml сервисе поддержана ручка metrics, для которой настроен мониторинг:

In [18]:
!curl http://localhost:8000/metrics

# HELP python_gc_objects_collected_total Objects collected during gc
# TYPE python_gc_objects_collected_total counter
python_gc_objects_collected_total{generation="0"} 574.0
python_gc_objects_collected_total{generation="1"} 400.0
python_gc_objects_collected_total{generation="2"} 20.0
# HELP python_gc_objects_uncollectable_total Uncollectable objects found during GC
# TYPE python_gc_objects_uncollectable_total counter
python_gc_objects_uncollectable_total{generation="0"} 0.0
python_gc_objects_uncollectable_total{generation="1"} 0.0
python_gc_objects_uncollectable_total{generation="2"} 0.0
# HELP python_gc_collections_total Number of times this generation was collected
# TYPE python_gc_collections_total counter
python_gc_collections_total{generation="0"} 581.0
python_gc_collections_total{generation="1"} 52.0
python_gc_collections_total{generation="2"} 4.0
# HELP python_info Python platform information
# TYPE python_info gauge
python_info{implementation="CPython",major="3",minor="11",patc

3. График метрики в prometheus:

<div style="text-align:center">
  <img src="assets/img_prometheus_graph.png" width="800">
</div>

4. Дешборд в graphana:

<div style="text-align:center">
  <img src="assets/img_dashboard.png" width="800">
</div>

5. Настроенный алерт:

<div style="text-align:center">
  <img src="assets/img_alert_firing.png" width="800">
</div>

5. Нотификация об алерте в tg:

<div style="text-align:center">
  <img src="assets/img_alert_contact.png" width="800">
</div>

## 3. Обнаружение дрифта данных

Для отслеживания дрифта данных была поддержаны api/drift ручки.

1. Проверим статус дрифта сразу после включения сервиса:

In [15]:
!curl http://localhost:8000/api/drift

{"drift_factor":1.0,"drift_score":0.0,"drifted_features_count":0,"drift_detected":false,"precision_clean":1.0,"precision_drifted":1.0,"last_run_at":null}

2. Включаем дрифт через специальную ручку:

In [16]:
!curl -X POST http://localhost:8000/api/drift/trigger

{"drift_factor":10.0,"drift_score":1.0,"drifted_features_count":4,"drift_detected":true,"precision_clean":1.0,"precision_drifted":0.1238,"last_run_at":"2026-05-17T12:17:20Z"}

3. Визуализация детекции дрифта в graphana:

<div style="text-align:center">
  <img src="assets/img_data_drift.png" width="800">
</div>

## 4. Обеспечение качества данных

В задании предлагается настроить мониторинг качества данных при помощи dqops. Был добавлен соответствующий image в docker-compose, при запуске получаем следующую ошибку:

```txt
(.venv) mishasdk@m1-pro ~/m/d/mipt-ml-monitoring (main)> docker run -it -p 8080:8080 -v $(pwd)/dqops_workspace:/dqo/userhome dqops/dqo

  ___     ___     ___
 |   \   / _ \   / _ \   _ __   ___
 | |) | | (_) | | (_) | | '_ \ (_-<
 |___/   \__\_\  \___/  | .__/ /__/
                        |_|
 :: DQOps Data Quality Operations Center ::    (v1.13.1)

DQOps no longer supports a FREE edition. Please contact DQOps sales at https://dqops.com/contact-us/ to obtain a valid trial key.
```

Тут говорится что dqps больше нельзя использовать бесплатно, поэтому у нас не получится использовать его для решения задания.